# Heart Disease Prediction – Neural Network with PyTorch

**Dataset:** [Heart Disease Dataset – Kaggle (johnsmith88)](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset/data)  
**Optimizer:** Adam  
**Task:** Binary classification – predict presence of heart disease

---
## Features
| Column | Description |
|--------|-------------|
| age | Age in years |
| sex | 1 = male, 0 = female |
| cp | Chest pain type (0–3) |
| trestbps | Resting blood pressure (mm Hg) |
| chol | Serum cholesterol (mg/dl) |
| fbs | Fasting blood sugar > 120 mg/dl (1/0) |
| restecg | Resting ECG results (0–2) |
| thalach | Max heart rate achieved |
| exang | Exercise-induced angina (1/0) |
| oldpeak | ST depression induced by exercise |
| slope | Slope of peak exercise ST segment (0–2) |
| ca | Major vessels colored by fluoroscopy (0–3) |
| thal | Thalassemia (0–2) |
| **target** | **1 = disease, 0 = no disease** |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from classes import HeartDiseaseDataset, HeartDiseaseModel, CrossEntropyLoss, Adam, FEATURE_COLS
from utils import (
    train_loop, test_loop,
    plot_loss_curves, plot_accuracy_curve,
    plot_confusion_matrix, count_parameters,
)

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"PyTorch version: {torch.__version__}")

## 2. Exploratory Data Analysis

In [ ]:
df = pd.read_csv('heart.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Class distribution
counts = df['target'].value_counts()
labels = ['No disease (0)', 'Disease (1)']
plt.figure(figsize=(5, 4))
plt.bar(labels, [counts[0], counts[1]], color=['steelblue', 'tomato'])
plt.title('Class distribution')
plt.ylabel('Count')
for i, v in enumerate([counts[0], counts[1]]):
    plt.text(i, v + 1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Class balance: {counts[1]/len(df)*100:.1f}% positive")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr().round(2), annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by target
cont_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, len(cont_features), figsize=(16, 4))
for ax, feat in zip(axes, cont_features):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        ax.hist(df[df['target'] == label][feat], bins=20,
                alpha=0.6, color=color, label=f'target={label}')
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('Continuous feature distributions by target')
plt.tight_layout()
plt.show()

## 3. Data Preparation

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_SEED, stratify=df['target']
)

train_dataset = HeartDiseaseDataset(train_df)
test_dataset  = HeartDiseaseDataset(test_df, scaler=train_dataset.scaler)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train: {len(train_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")

X_sample, y_sample = next(iter(train_loader))
print(f"Batch X shape: {X_sample.shape}")
print(f"Batch y shape: {y_sample.shape}")

## 4. Baseline Model

In [ ]:
torch.manual_seed(RANDOM_SEED)

model   = HeartDiseaseModel(dropout=0.3)
print(model)
print(f"Trainable parameters: {count_parameters(model):,}")

In [ ]:
EPOCHS = 60
LR     = 1e-3

loss_fn   = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR)

train_losses, test_losses, accuracies = [], [], []

for epoch in range(1, EPOCHS + 1):
    tr_loss        = train_loop(train_loader, model, loss_fn, optimizer)
    te_loss, acc   = test_loop(test_loader,   model, loss_fn)

    train_losses.append(tr_loss)
    test_losses.append(te_loss)
    accuracies.append(acc)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3d}/{EPOCHS}  "
              f"Train: {tr_loss:.4f}  "
              f"Test: {te_loss:.4f}  "
              f"Acc: {acc:.1f}%")

print(f"\nBest accuracy: {max(accuracies):.1f}%")

In [ ]:
plot_loss_curves(train_losses, test_losses, title='Baseline model – Loss curves (Adam)')
plot_accuracy_curve(accuracies, title='Baseline model – Test accuracy')

In [ ]:
plot_confusion_matrix(model, test_loader)

## 5. Experiment: Effect of Learning Rate

In [ ]:
lr_results = {}
EPOCHS_EXP = 60

for lr in [1e-2, 1e-3, 5e-4, 1e-4]:
    torch.manual_seed(RANDOM_SEED)
    m   = HeartDiseaseModel(dropout=0.3)
    opt = Adam(m.parameters(), lr=lr)
    lf  = CrossEntropyLoss()

    accs = []
    for _ in range(EPOCHS_EXP):
        train_loop(train_loader, m, lf, opt)
        _, acc = test_loop(test_loader, m, lf)
        accs.append(acc)

    lr_results[lr] = accs
    print(f"lr={lr:.0e}  best acc={max(accs):.1f}%  final acc={accs[-1]:.1f}%")

plt.figure(figsize=(9, 4))
for lr, accs in lr_results.items():
    plt.plot(range(1, EPOCHS_EXP + 1), accs, label=f'lr={lr:.0e}')
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('Learning rate comparison – Adam')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Experiment: Effect of Dropout

In [ ]:
dropout_results = {}

for dropout in [0.0, 0.2, 0.3, 0.5]:
    torch.manual_seed(RANDOM_SEED)
    m   = HeartDiseaseModel(dropout=dropout)
    opt = Adam(m.parameters(), lr=1e-3)
    lf  = CrossEntropyLoss()

    accs = []
    for _ in range(EPOCHS_EXP):
        train_loop(train_loader, m, lf, opt)
        _, acc = test_loop(test_loader, m, lf)
        accs.append(acc)

    dropout_results[dropout] = accs
    print(f"dropout={dropout}  best acc={max(accs):.1f}%  final acc={accs[-1]:.1f}%")

plt.figure(figsize=(9, 4))
for d, accs in dropout_results.items():
    plt.plot(range(1, EPOCHS_EXP + 1), accs, label=f'dropout={d}')
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('Dropout comparison – Adam')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Experiment: Adam vs SGD

In [ ]:
optimizer_results = {}

for name, build_opt in [
    ('Adam (lr=1e-3)',  lambda p: Adam(p, lr=1e-3)),
    ('Adam (lr=5e-4)',  lambda p: Adam(p, lr=5e-4)),
    ('SGD  (lr=1e-2)',  lambda p: torch.optim.SGD(p, lr=1e-2, momentum=0.9)),
    ('SGD  (lr=3e-3)',  lambda p: torch.optim.SGD(p, lr=3e-3, momentum=0.9)),
]:
    torch.manual_seed(RANDOM_SEED)
    m   = HeartDiseaseModel(dropout=0.3)
    opt = build_opt(m.parameters())
    lf  = CrossEntropyLoss()

    accs = []
    for _ in range(EPOCHS_EXP):
        train_loop(train_loader, m, lf, opt)
        _, acc = test_loop(test_loader, m, lf)
        accs.append(acc)

    optimizer_results[name] = accs
    print(f"{name:<22s}  best acc={max(accs):.1f}%  final acc={accs[-1]:.1f}%")

plt.figure(figsize=(9, 4))
for name, accs in optimizer_results.items():
    ls = '-' if 'Adam' in name else '--'
    plt.plot(range(1, EPOCHS_EXP + 1), accs, linestyle=ls, label=name)
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('Custom Adam vs SGD')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Final Model – Best Configuration

In [ ]:
# Train with best hyperparameters from the experiments above
BEST_LR      = 1e-3
BEST_DROPOUT = 0.3
BEST_EPOCHS  = 80

torch.manual_seed(RANDOM_SEED)
best_model = HeartDiseaseModel(dropout=BEST_DROPOUT)
loss_fn    = CrossEntropyLoss()
optimizer  = Adam(best_model.parameters(), lr=BEST_LR)

tr_losses, te_losses, accs = [], [], []

for epoch in range(1, BEST_EPOCHS + 1):
    trl        = train_loop(train_loader, best_model, loss_fn, optimizer)
    tel, acc   = test_loop(test_loader,   best_model, loss_fn)
    tr_losses.append(trl)
    te_losses.append(tel)
    accs.append(acc)

print(f"Best accuracy: {max(accs):.1f}%  (epoch {accs.index(max(accs))+1})")
print(f"Final accuracy: {accs[-1]:.1f}%")

plot_loss_curves(tr_losses, te_losses, title='Final model – Loss curves')
plot_accuracy_curve(accs, title='Final model – Test accuracy')

In [ ]:
plot_confusion_matrix(best_model, test_loader)

## 9. Save Final Model

In [ ]:
torch.save({
    'model_state': best_model.state_dict(),
    'scaler':      train_dataset.scaler,
}, 'heart_model.pth')

print('Model saved to heart_model.pth')
print('Run app.py to use it interactively.')